# 🚀 Self-Hosted AI API on Google Colab

This notebook sets up a complete AI API endpoint on Google Colab's free GPU.

## What this does:
1. Installs Ollama (LLM runtime)
2. Pulls an open-source model (Qwen 1.8B)
3. Starts FastAPI server
4. Exposes via ngrok for public access

## ⚠️ Important Notes:
- Colab sessions last ~12 hours maximum
- GPU availability varies on free tier
- ngrok URL changes each session
- Keep this tab open to maintain the session

## Step 1: Install Dependencies

In [ ]:
#@title Install Ollama and Python Dependencies
#@markdown Run this cell to install all required packages

import subprocess
import sys

print("📦 Installing system dependencies...")

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install Python packages
!pip install fastapi uvicorn pyngrok nest_asyncio requests python-dotenv -q

print("✅ Installation complete!")

## Step 2: Configure ngrok

In [ ]:
#@title Enter your ngrok Auth Token
#@markdown Get your token from https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # @param {type: "string"}

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print(f"✅ ngrok token configured")

## Step 3: Start Ollama Server

In [ ]:
#@title Start Ollama Server
#@markdown This starts the Ollama service in the background

import subprocess
import time

# Kill any existing Ollama processes
!pkill ollama
time.sleep(1)

# Start Ollama server in background
print("🚀 Starting Ollama server...")
!nohup bash -c "OLLAMA_HOST=0.0.0.0 ollama serve" > ollama.log 2>&1 &

# Wait for server to start
print("⏳ Waiting for Ollama to start...")
time.sleep(5)

# Verify it's running
!curl -s http://127.0.0.1:11434 && echo "\n✅ Ollama is running!" || echo "❌ Ollama failed to start"

## Step 4: Pull AI Model

In [ ]:
#@title Download AI Model
#@markdown Choose which model to use. Qwen 1.8B is recommended for Colab's free tier.

MODEL_NAME = "qwen:1.8b"  # @param ["qwen:1.8b", "qwen:7b", "llama2", "mistral", "phi", "gemma"]

print(f"📥 Pulling model: {MODEL_NAME}")
print("This may take a few minutes depending on your internet connection...")

!ollama pull $MODEL_NAME

print(f"✅ Model {MODEL_NAME} ready!")

## Step 5: Create the FastAPI App

In [ ]:
#@title Create FastAPI Application
#@markdown This creates the app.py file with the API code

app_code = '''
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
import requests
import os
import time

app = FastAPI(
    title="Self-Hosted AI API",
    description="OpenAI-compatible chat completions endpoint",
    version="1.0.0"
)

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434/api/generate")
DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "qwen:1.8b")


class Message(BaseModel):
    role: str
    content: str


class ChatCompletionRequest(BaseModel):
    messages: List[Message]
    model: Optional[str] = DEFAULT_MODEL
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = None
    stream: Optional[bool] = False


class ChatCompletionChoice(BaseModel):
    index: int = 0
    message: Message
    finish_reason: Optional[str] = "stop"


class ChatCompletionResponse(BaseModel):
    id: str = "chatcmpl-1"
    object: str = "chat.completion"
    created: int = 0
    model: str
    choices: List[ChatCompletionChoice]
    usage: Optional[Dict[str, int]] = None


@app.get("/health")
async def health_check():
    return {"status": "healthy", "ollama_url": OLLAMA_URL}


@app.get("/v1/models")
async def list_models():
    return {
        "object": "list",
        "data": [
            {
                "id": DEFAULT_MODEL,
                "object": "model",
                "created": 1234567890,
                "owned_by": "ollama"
            }
        ]
    }


@app.post("/v1/chat/completions", response_model=ChatCompletionResponse)
async def chat_completions(request: ChatCompletionRequest):
    try:
        latest_message = request.messages[-1]
        prompt = latest_message.content

        conversation = "\\n\\n".join([
            f"{msg.role}: {msg.content}" for msg in request.messages
        ])

        response = requests.post(
            OLLAMA_URL,
            json={
                "model": request.model,
                "prompt": conversation,
                "stream": False,
                "options": {
                    "temperature": request.temperature,
                    "num_predict": request.max_tokens or 512
                }
            },
            timeout=120
        )

        if response.status_code != 200:
            raise HTTPException(
                status_code=502,
                detail=f"Ollama error: {response.status_code}"
            )

        result = response.json()
        assistant_message = result.get("response", "")

        return ChatCompletionResponse(
            model=request.model,
            choices=[
                ChatCompletionChoice(
                    message=Message(
                        role="assistant",
                        content=assistant_message
                    )
                )
            ],
            usage={
                "prompt_tokens": len(conversation.split()) // 4,
                "completion_tokens": len(assistant_message.split()) // 4,
                "total_tokens": (len(conversation.split()) + len(assistant_message.split())) // 4
            }
        )

    except requests.exceptions.ConnectionError:
        raise HTTPException(
            status_code=503,
            detail="Cannot connect to Ollama. Ensure it is running."
        )
    except requests.exceptions.Timeout:
        raise HTTPException(
            status_code=504,
            detail="Ollama request timed out"
        )
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created!")

## Step 6: Start the API Server

In [ ]:
#@title Start FastAPI Server
#@markdown This starts the API server on port 8000

import subprocess
import time

print("🚀 Starting FastAPI server...")

# Start uvicorn in background
!nohup python app.py > api.log 2>&1 &

# Wait for server to start
print("⏳ Waiting for server to start...")
time.sleep(3)

# Verify it's running
!curl -s http://localhost:8000/health && echo "\n✅ FastAPI is running on port 8000!" || echo "❌ Server failed to start"

## Step 7: Expose via ngrok

In [ ]:
#@title Create Public URL with ngrok
#@markdown This exposes your local API to the internet

from pyngrok import ngrok

# Create tunnel
public_url = ngrok.connect(8000)

print("\n" + "="*60)
print("🎉 YOUR AI API IS NOW PUBLIC!")
print("="*60)
print(f"\n📡 Public URL: {public_url}")
print(f"\n📝 API Endpoint: {public_url}/v1/chat/completions")
print(f"💚 Health Check: {public_url}/health")
print("\n" + "="*60)
print("⚠️  Keep this notebook running to maintain the URL")
print("⚠️  Colab sessions expire after ~12 hours")
print("="*60)

## Step 8: Test Your API

In [ ]:
#@title Test the API
#@markdown Run this cell to test your new AI API

import requests

# Get the ngrok URL from the previous cell
NGROK_URL = str(public_url)  # This will be set automatically

print(f"Testing API at: {NGROK_URL}")

# Test health endpoint
print("\n📋 Testing health endpoint...")
health = requests.get(f"{NGROK_URL}/health")
print(f"Health: {health.json()}")

# Test chat endpoint
print("\n💬 Testing chat completions...")
response = requests.post(
    f"{NGROK_URL}/v1/chat/completions",
    json={
        "messages": [
            {"role": "user", "content": "Hello! Introduce yourself in one sentence."}
        ]
    }
)

if response.status_code == 200:
    result = response.json()
    print(f"✅ Response: {result['choices'][0]['message']['content']}")
else:
    print(f"❌ Error: {response.status_code} - {response.text}")

## Step 9: Use with OpenAI SDK

In [ ]:
#@title Example: OpenAI SDK Compatibility
#@markdown Your API is compatible with the OpenAI Python SDK!

# First, install the openai package
!pip install openai -q

from openai import OpenAI

# Initialize client with your self-hosted endpoint
client = OpenAI(
    base_url=f"{str(public_url)}/v1",
    api_key="not-needed"  # No API key required for self-hosted
)

# Chat with your model
response = client.chat.completions.create(\n    model="qwen:1.8b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the meaning of life?"}
    ]
)

print(f"\n🤖 AI Response: {response.choices[0].message.content}")

---

## 📊 Session Info

In [ ]:
#@title Check GPU and System Info

!nvidia-smi

print("\n" + "="*60)
print("📦 Installed Models:")
!ollama list

print("\n" + "="*60)
print("📋 API Logs (last 20 lines):")
!tail -20 api.log